# Convertidor de imágenes a PDF

Selecciona una carpeta para generar un PDF con una imagen por página. Se conservan todos los píxeles: no se remuestrean imágenes ni se vuelve a comprimir en JPEG. Las imágenes transparentes se representan sobre fondo blanco.

Ejecuta las celdas en orden. La aplicación está pensada para Jupyter local; el selector usa el explorador nativo del equipo donde se ejecuta el kernel. Si el entorno no puede abrirlo, puedes pegar la ruta manualmente.

## 1. Comprobación e instalación de dependencias

Instala Pillow, ReportLab e ipywidgets solo si falta alguno.

In [1]:
import importlib.util
import subprocess
import sys

DEPENDENCIES = {
    "PIL": "Pillow",
    "reportlab": "reportlab",
    "ipywidgets": "ipywidgets",
}
missing = [package for module, package in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Instalando dependencias faltantes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Todas las dependencias necesarias ya están instaladas.")

Todas las dependencias necesarias ya están instaladas.


## 2. Funciones de búsqueda y generación del PDF

La lista de formatos se puede ampliar editando `SUPPORTED_FORMATS`. Cada página adopta la proporción y orientación de su imagen.

In [2]:
from collections import Counter
from pathlib import Path
import html
import os
import subprocess
import sys
import tempfile
import threading
import tkinter as tk
from tkinter import filedialog

import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageOps
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

SUPPORTED_FORMATS = {
    ".jpg": "JPEG", ".jpeg": "JPEG", ".png": "PNG", ".webp": "WEBP",
    ".bmp": "BMP", ".tif": "TIFF", ".tiff": "TIFF", ".gif": "GIF (primer fotograma)", ".ico": "ICO",
}
PDF_MAX_PAGE_POINTS = 14_400  # Límite práctico de tamaño de página en lectores PDF.

def find_images(folder: Path, recursive: bool, alphabetical: bool) -> list[Path]:
    """Devuelve archivos con extensiones compatibles; la lectura real se valida al convertir."""
    candidates = folder.rglob("*") if recursive else folder.iterdir()
    images = [item for item in candidates if item.is_file() and item.suffix.lower() in SUPPORTED_FORMATS]
    if alphabetical:
        images.sort(key=lambda item: (item.name.casefold(), item.name))
    return images

def valid_pdf_name(raw_name: str) -> tuple[str | None, str | None]:
    name = raw_name.strip()
    if name.lower().endswith(".pdf"):
        name = name[:-4]
    if not name:
        return None, "Escribe un nombre para el documento PDF."
    if any(ord(char) < 32 or char in '<>:\"/\\|?*' for char in name):
        return None, "El nombre contiene caracteres no válidos para un archivo."
    if name.endswith((".", " ")):
        return None, "El nombre no puede terminar en punto o espacio."
    if name.split(".")[0].upper() in {"CON", "PRN", "AUX", "NUL", *(f"COM{i}" for i in range(1, 10)), *(f"LPT{i}" for i in range(1, 10))}:
        return None, "Ese nombre está reservado por el sistema operativo."
    return f"{name}.pdf", None

def unique_output_path(folder: Path, filename: str) -> Path:
    candidate = folder / filename
    if not candidate.exists():
        return candidate
    stem = Path(filename).stem
    for number in range(1, 10_000):
        candidate = folder / f"{stem} ({number}).pdf"
        if not candidate.exists():
            return candidate
    raise FileExistsError("No se encontró un nombre de salida libre después de 9999 intentos.")

def page_dimensions(pixel_width: int, pixel_height: int, dpi: tuple[float, float]) -> tuple[float, float]:
    dpi_x = dpi[0] if dpi and dpi[0] and dpi[0] > 0 else 150.0
    dpi_y = dpi[1] if dpi and len(dpi) > 1 and dpi[1] and dpi[1] > 0 else dpi_x
    width = pixel_width * 72.0 / dpi_x
    height = pixel_height * 72.0 / dpi_y
    scale = min(1.0, PDF_MAX_PAGE_POINTS / max(width, height))
    return width * scale, height * scale

def draw_image_page(pdf: canvas.Canvas, image_path: Path) -> None:
    """Draw one full-resolution image on a page; never resize its pixel data."""
    with Image.open(image_path) as source:
        if source.format == "GIF":
            source.seek(0)
        dpi = source.info.get("dpi", (150.0, 150.0))
        oriented = ImageOps.exif_transpose(source)
        oriented.load()
        page_width, page_height = page_dimensions(oriented.width, oriented.height, dpi)
        if "A" in oriented.getbands() or "transparency" in oriented.info:
            rgba = oriented.convert("RGBA")
            rgb = Image.new("RGB", rgba.size, "white")
            rgb.paste(rgba, mask=rgba.getchannel("A"))
            rgba.close()
        else:
            rgb = oriented.convert("RGB")
        try:
            page.setPageSize((page_width, page_height))
            page.drawImage(ImageReader(rgb), 0, 0, width=page_width, height=page_height, preserveAspectRatio=True)
            page.showPage()
        finally:
            rgb.close()

def convert_images(folder: Path, images: list[Path], filename: str, cancel_event: threading.Event, progress_callback):
    """Create a PDF without overwriting an existing file; unreadable images are reported and skipped."""
    pdf_path = unique_output_path(folder, filename)
    pdf_stream = None
    created = False
    processed = 0
    errors = []
    pdf = None
    try:
        for _ in range(10_000):
            pdf_path = unique_output_path(folder, filename)
            try:
                pdf_stream = open(pdf_path, "xb")
                created = True
                break
            except FileExistsError:
                continue
        if pdf_stream is None:
            raise FileExistsError("No se pudo reservar un nombre de PDF sin sobrescribir otro archivo.")
        pdf = canvas.Canvas(pdf_stream, pageCompression=1)
        for index, image_path in enumerate(images, start=1):
            if cancel_event.is_set():
                break
            try:
                draw_image_page(pdf, image_path)
                processed += 1
            except Exception as exc:
                errors.append((image_path.name, f"{type(exc).__name__}: {exc}"))
            progress_callback(index, len(images), image_path.name)
        pdf.save()
        pdf_stream.close()
        pdf_stream = None
        if cancel_event.is_set() or processed == 0:
            pdf_path.unlink(missing_ok=True)
            created = False
        return pdf_path, processed, errors, cancel_event.is_set()
    except Exception:
        if pdf_stream is not None:
            pdf_stream.close()
        if created:
            pdf_path.unlink(missing_ok=True)
        raise

def format_size(size: int) -> str:
    value = float(size)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if value < 1024 or unit == "TB":
            return f"{value:.1f} {unit}" if unit != "B" else f"{int(value)} B"
        value /= 1024
    return f"{value:.1f} TB"

print("Funciones listas. Continúa a la interfaz gráfica.")

Funciones listas. Continúa a la interfaz gráfica.


## 3. Interfaz de conversión

La salida se guarda en la misma carpeta de origen. Si el nombre ya existe, se conserva el documento anterior y se propone automáticamente un nombre como `documento (1).pdf`.

In [3]:
folder_input = widgets.Text(value="", description="Carpeta", placeholder="Ruta de la carpeta de imágenes", layout=widgets.Layout(width="100%"))
name_input = widgets.Text(value="imagenes_combinadas", description="Nombre PDF", layout=widgets.Layout(width="100%"))
recursive_option = widgets.Checkbox(value=False, description="Buscar también en subcarpetas")
alphabetical_option = widgets.Checkbox(value=True, description="Ordenar alfabéticamente por nombre")
pick_button = widgets.Button(description="Seleccionar carpeta", icon="folder-open", button_style="")
apply_path_button = widgets.Button(description="Usar ruta", icon="check")
convert_button = widgets.Button(description="CONVERTIR A PDF", button_style="primary", icon="file-pdf")
cancel_button = widgets.Button(description="Cancelar", button_style="warning", disabled=True, icon="stop")
open_folder_button = widgets.Button(description="Abrir carpeta de salida", icon="external-link-alt", disabled=True)
progress_bar = widgets.IntProgress(value=0, min=0, max=1, description="Progreso:", bar_style="info", layout=widgets.Layout(width="100%"))
preview = widgets.HTML()
status = widgets.HTML(value="<span style='color:#64748b'>Selecciona una carpeta para ver el resumen previo.</span>")
result = widgets.Output()
_cancel_event = threading.Event()
_running = False
_last_output = None

def _folder_from_input() -> Path | None:
    raw = folder_input.value.strip().strip('"')
    if not raw:
        return None
    return Path(raw).expanduser()

def refresh_preview(_=None) -> None:
    folder = _folder_from_input()
    filename, name_error = valid_pdf_name(name_input.value)
    if folder is None or not folder.is_dir():
        preview.value = "<div style='padding:12px;background:#f8fafc;border-radius:8px;color:#64748b'>Indica o selecciona una carpeta válida.</div>"
        return
    try:
        images = find_images(folder, recursive_option.value, alphabetical_option.value)
    except OSError as exc:
        preview.value = f"<div style='color:#b91c1c'>No se pudo leer la carpeta: {html.escape(str(exc))}</div>"
        return
    counts = Counter(SUPPORTED_FORMATS[item.suffix.lower()] for item in images)
    detected = ", ".join(f"{html.escape(kind)}: {count}" for kind, count in sorted(counts.items())) or "Ninguno"
    proposed = "Nombre inválido" if name_error else unique_output_path(folder, filename).name
    preview.value = (
        "<div style='padding:14px 16px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;line-height:1.8'>"
        f"<b>Imágenes encontradas:</b> {len(images)}<br>"
        f"<b>Formatos:</b> {detected}<br>"
        f"<b>PDF que se generará:</b> {html.escape(proposed)}<br>"
        f"<b>Ubicación:</b> {html.escape(str(folder))}</div>"
    )

def on_pick_folder(_):
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askdirectory(parent=root, title="Seleccionar carpeta con imágenes")
        root.destroy()
        if selected:
            folder_input.value = selected
            status.value = "<span style='color:#15803d'>Carpeta seleccionada.</span>"
            refresh_preview()
    except Exception as exc:
        status.value = f"<span style='color:#b91c1c'>No se pudo abrir el explorador nativo ({html.escape(str(exc))}). Pega la ruta en el campo Carpeta y pulsa Usar ruta.</span>"

def on_apply_path(_):
    folder = _folder_from_input()
    if folder is None or not folder.is_dir():
        status.value = "<span style='color:#b91c1c'>La ruta indicada no existe o no es una carpeta.</span>"
        refresh_preview()
        return
    folder_input.value = str(folder.resolve())
    status.value = "<span style='color:#15803d'>Ruta aplicada.</span>"
    refresh_preview()

def on_cancel(_):
    if _running:
        _cancel_event.set()
        status.value = "<span style='color:#a16207'>Cancelando de forma segura...</span>"

def on_convert(_):
    global _running, _last_output
    if _running:
        return
    folder = _folder_from_input()
    filename, name_error = valid_pdf_name(name_input.value)
    if folder is None or not folder.is_dir():
        status.value = "<span style='color:#b91c1c'>Selecciona una carpeta válida antes de convertir.</span>"
        return
    if name_error:
        status.value = f"<span style='color:#b91c1c'>{html.escape(name_error)}</span>"
        return
    try:
        images = find_images(folder, recursive_option.value, alphabetical_option.value)
    except OSError as exc:
        status.value = f"<span style='color:#b91c1c'>No se pudo leer la carpeta: {html.escape(str(exc))}</span>"
        return
    if not images:
        status.value = "<span style='color:#b91c1c'>No se encontraron imágenes compatibles en la carpeta.</span>"
        refresh_preview()
        return
    _running = True
    _cancel_event.clear()
    _last_output = None
    progress_bar.value = 0
    progress_bar.max = len(images)
    convert_button.disabled = True
    cancel_button.disabled = False
    open_folder_button.disabled = True
    status.value = f"<span style='color:#334155'>Preparando {len(images)} imágenes...</span>"
    refresh_preview()

    def update_progress(current, total, current_name):
        progress_bar.value = current
        status.value = f"<span style='color:#334155'>Procesando {current} de {total}: {html.escape(current_name)}</span>"

    def worker():
        global _running, _last_output
        try:
            output_path, processed, errors, cancelled = convert_images(folder, images, filename, _cancel_event, update_progress)
            if cancelled:
                status.value = "<span style='color:#a16207'>Conversión cancelada. Se eliminó el PDF incompleto.</span>"
            elif processed == 0:
                status.value = "<span style='color:#b91c1c'>No se pudo procesar ninguna imagen. Revisa los archivos y sus permisos.</span>"
            else:
                _last_output = output_path
                summary = (
                    f"<div style='padding:14px 16px;background:#f0fdf4;border:1px solid #bbf7d0;border-radius:10px;line-height:1.8;color:#166534'>"
                    f"<b>Conversión finalizada correctamente</b><br>PDF: {html.escape(output_path.name)}<br>"
                    f"Ubicación: {html.escape(str(output_path))}<br>Imágenes procesadas: {processed}<br>"
                    f"Tamaño: {format_size(output_path.stat().st_size)}</div>"
                )
                if errors:
                    error_items = "".join(f"<li>{html.escape(name)} — {html.escape(message)}</li>" for name, message in errors[:10])
                    extra = f"<p><b>Archivos omitidos ({len(errors)}):</b><ul>{error_items}</ul></p>"
                    if len(errors) > 10:
                        extra += f"<p>Y {len(errors) - 10} errores adicionales.</p>"
                    summary = summary.replace("</div>", f"{extra}</div>")
                status.value = summary
                open_folder_button.disabled = False
                refresh_preview()
        except Exception as exc:
            status.value = f"<span style='color:#b91c1c'>Error al crear el PDF: {html.escape(type(exc).__name__)}: {html.escape(str(exc))}</span>"
        finally:
            _running = False
            convert_button.disabled = False
            cancel_button.disabled = True

    threading.Thread(target=worker, name="image-pdf-converter", daemon=True).start()

def on_open_folder(_):
    if _last_output is None or not _last_output.exists():
        status.value = "<span style='color:#b91c1c'>No se encuentra el PDF generado.</span>"
        return
    try:
        if sys.platform == "win32":
            os.startfile(str(_last_output.parent))
        elif sys.platform == "darwin":
            subprocess.Popen(["open", str(_last_output.parent)])
        else:
            subprocess.Popen(["xdg-open", str(_last_output.parent)])
    except (OSError, AttributeError) as exc:
        status.value = f"<span style='color:#b91c1c'>No se pudo abrir la carpeta: {html.escape(str(exc))}</span>"

pick_button.on_click(on_pick_folder)
apply_path_button.on_click(on_apply_path)
convert_button.on_click(on_convert)
cancel_button.on_click(on_cancel)
open_folder_button.on_click(on_open_folder)
for control in (folder_input, name_input, recursive_option, alphabetical_option):
    control.observe(refresh_preview, names="value")

display(widgets.HTML("<h2 style='margin-bottom:4px'>Imágenes → PDF</h2><p style='color:#64748b;margin-top:0'>Un PDF nítido, una imagen por página.</p>"))
display(widgets.HBox([pick_button, apply_path_button]))
display(folder_input, name_input)
display(widgets.VBox([alphabetical_option, recursive_option]))
display(widgets.HTML("<b>Diseño de páginas</b><br>Orientación automática · proporciones originales · una imagen por página · sin recortes"))
display(preview, widgets.HBox([convert_button, cancel_button]), progress_bar, status, open_folder_button)
refresh_preview()

HTML(value="<h2 style='margin-bottom:4px'>Imágenes → PDF</h2><p style='color:#64748b;margin-top:0'>Un PDF níti…

Text(value='', description='Carpeta', layout=Layout(width='100%'), placeholder='Ruta de la carpeta de imágenes…

Text(value='imagenes_combinadas', description='Nombre PDF', layout=Layout(width='100%'))

HTML(value='<b>Diseño de páginas</b><br>Orientación automática · proporciones originales · una imagen por pági…

HTML(value='')

IntProgress(value=0, bar_style='info', description='Progreso:', layout=Layout(width='100%'), max=1)

HTML(value="<span style='color:#64748b'>Selecciona una carpeta para ver el resumen previo.</span>")

Button(description='Abrir carpeta de salida', disabled=True, icon='external-link-alt', style=ButtonStyle())

## Uso y notas

1. Ejecuta las celdas desde el principio.
2. Pulsa **Seleccionar carpeta** y elige la carpeta con las imágenes. Si Jupyter se ejecuta en otro equipo o no permite abrir el explorador, pega la ruta completa y pulsa **Usar ruta**.
3. Define el nombre, las opciones de orden y subcarpetas, y revisa el resumen previo.
4. Pulsa **CONVERTIR A PDF**. Los archivos dañados se omiten, se enumeran al final y no detienen el resto de la conversión.

El tamaño físico de página usa los DPI de la imagen si están disponibles y 150 DPI como valor predeterminado. Si un tamaño excediera el límite práctico de un lector PDF, solo se limita el tamaño físico de página: los píxeles de la imagen no se reducen. TIFF animados y GIF usan su primer fotograma.